# 02 — Pipeline d'import : dossier → base de données

Ce notebook couvre le chemin **fichiers → traitement → sauvegarde en base**.
C'est le workflow quotidien : après chaque mesure, on importe le fichier `.txt`
exporté depuis HRV Elite / Polar Flow et on enrichit la base avec les résultats.

**Architecture du pipeline :**
```
Dossier de fichiers
        ↓ parse_rr_file()       ← un seul parseur à changer pour Garmin/Apple Health
RRSeries + nettoyage
        ↓ protocole HRV (resting / orthostatic / …)
Résultats + score
        ↓ HRVRepository.save_*()
Base PostgreSQL
```

**Prérequis :** notebook `00_setup` exécuté (tables créées).

In [1]:
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path("../.env"), override=True)

import cardiolab
from cardiolab.analytics import Baseline, readiness_score_oura
from cardiolab.database import HRVRepository
from cardiolab.protocols import resting_hrv
from cardiolab.signals.rr import RRSeries

DATASETS_DIR = Path(cardiolab.__file__).parent / "datasets"

# Identifiant utilisateur — remplacer par votre UUID en production
USER_ID_TEST = "demo-user"

print("✓  Imports OK")
print(f"   Datasets : {DATASETS_DIR}")

✓  Imports OK
   Datasets : /home/jmilon/Projet-Pro-DevSecOps/Programmation/GitLab/cardioanalysis/cardiolab/cardiolab/datasets


## 1 — Choisir le parseur selon la source capteur

Le parseur est le **seul point de variation** selon le type de capteur.
Tout le reste du pipeline (nettoyage, protocole, sauvegarde) reste identique.

In [2]:
# ── Choisir le parseur ────────────────────────────────────────────────────────
#
# Aujourd'hui (v0.2.x) — Polar HRV Elite / H10
from cardiolab.sensors_tools.polar import parse_rr_file as parser

RESTING_DIR = DATASETS_DIR / "raw" / "resting"

# Demain (v0.3.0) — décommenter la ligne correspondante :
# from cardiolab.sensors_tools.garmin import parse_garmin_fit as parser
# from cardiolab.sensors_tools.apple_health import parse_apple_health_export as parser
# from cardiolab.sensors_tools.hrv4training import parse_hrv4training_csv as parser

files = sorted(RESTING_DIR.glob("*.txt"))
print(f"Fichiers trouvés : {len(files)}")
for f in files:
    print(f"  {f.name}")

Fichiers trouvés : 6
  2026-04-24 07-52-36.txt
  2026-04-25 07-58-25.txt
  2026-04-26 07-40-41.txt
  2026-04-27 07-01-19.txt
  2026-04-28 07-01-05.txt
  2026-04-30 07-01-38.txt


## 2 — Parcourir et traiter chaque fichier

Pour chaque fichier :
1. Parser → `RRSeries` nettoyée
2. Appliquer le protocole resting HRV
3. Calculer le score de readiness (basé sur la baseline progressive)
4. Sauvegarder features + RR bruts en base

In [3]:
results_list = []
dates_list   = []

with HRVRepository.from_env() as repo:
    existing = repo.load_features(USER_ID_TEST)
    baseline = Baseline.from_features(existing) if existing else Baseline()

    for filepath in files:
        # Extraire la date depuis le nom de fichier (format : YYYY-MM-DD HH-MM-SS.txt)
        date_str = filepath.stem[:10]  # "2026-04-24"

        # ── 1. Lire + nettoyer ────────────────────────────────────────────────
        raw  = parser(filepath)
        rr   = RRSeries(raw["rr_intervals"]).remove_outliers()

        # ── 2. Protocole HRV ──────────────────────────────────────────────────
        features = resting_hrv(rr, compute_score=True)
        features.date = date_str  # save_features lit features.date

        # ── 3. Score readiness ────────────────────────────────────────────────
        score = readiness_score_oura(features, baseline)
        features.score = score

        # ── 4. Sauvegarde en base ─────────────────────────────────────────────
        repo.save_features(features, user_id=USER_ID_TEST)
        repo.save_raw_session(
            rr, user_id=USER_ID_TEST, date=date_str,
            protocol="resting", source_file=filepath.name,
        )

        # Enrichir la baseline pour la session suivante
        baseline.history.append(features)

        results_list.append(features)
        dates_list.append(date_str)
        print(f"  ✓  {date_str}  RMSSD {features.rmssd:.1f} ms  "
              f"HR {features.mean_hr:.1f} bpm  score {score:.0f}/100")

print(f"\n✅  {len(results_list)} sessions importées.")

  ✓  2026-04-24  RMSSD 78.5 ms  HR 57.9 bpm  score 50/100
  ✓  2026-04-25  RMSSD 51.8 ms  HR 52.2 bpm  score 37/100
  ✓  2026-04-26  RMSSD 53.5 ms  HR 55.3 bpm  score 38/100
  ✓  2026-04-27  RMSSD 52.6 ms  HR 53.5 bpm  score 51/100
  ✓  2026-04-28  RMSSD 59.9 ms  HR 53.9 bpm  score 60/100
  ✓  2026-04-30  RMSSD 81.9 ms  HR 51.0 bpm  score 83/100

✅  6 sessions importées.


## 3 — Vérifier en base

Relire depuis la base pour confirmer que l'import s'est bien passé.

In [4]:
with HRVRepository.from_env() as repo:
    saved = repo.load_features(USER_ID_TEST)

print(f"Sessions en base pour '{USER_ID_TEST}' : {len(saved)}")
print(f"\n{'Date':<12} {'RMSSD':>8} {'HR':>7} {'Score':>7}")
print("-" * 38)
for f in saved:
    print(f"  {str(f.date):<10}  {f.rmssd:>6.1f} ms  {f.mean_hr:>5.1f} bpm  {f.score:>5.0f}/100")

Sessions en base pour 'demo-user' : 6

Date            RMSSD      HR   Score
--------------------------------------
  2026-04-24    78.5 ms   57.9 bpm     50/100
  2026-04-25    51.8 ms   52.2 bpm     37/100
  2026-04-26    53.5 ms   55.3 bpm     38/100
  2026-04-27    52.6 ms   53.5 bpm     51/100
  2026-04-28    59.9 ms   53.9 bpm     60/100
  2026-04-30    81.9 ms   51.0 bpm     83/100


## 4 — Importer un fichier orthostatique

Même pipeline, protocole différent. Un fichier orthostatique disponible dans les datasets.

In [5]:
from cardiolab.protocols import orthostatic_hrv

ORTHO_DIR = DATASETS_DIR / "raw" / "orthostatic"
ortho_files = sorted(ORTHO_DIR.glob("*.txt"))

with HRVRepository.from_env() as repo:
    for filepath in ortho_files:
        date_str = filepath.stem[:10]

        raw      = parser(filepath)
        rr       = RRSeries(raw["rr_intervals"])
        result   = orthostatic_hrv(rr)

        repo.save_orthostatic(result, user_id=USER_ID_TEST, date=date_str)
        repo.save_raw_session(
            rr, user_id=USER_ID_TEST, date=date_str,
            protocol="orthostatic", source_file=filepath.name,
        )
        print(f"  ✓  {date_str}  interp={result.interpretation}  "
              f"ΔHR={result.hr_response:.1f} bpm  score={result.score:.0f}/100")

print("\n✅  Import orthostatique terminé.")

/home/jmilon/Projet-Pro-DevSecOps/Programmation/GitLab/cardioanalysis/cardiolab/cardiolab/protocols/orthostatic.py:500: PhysiologicalWarning: Recording duration (20 s) is below the recommended minimum of 300 s for resting HRV analysis. Frequency-domain and non-linear metrics may be unreliable.
  features=resting_hrv(transition_rr, method=method),
/home/jmilon/Projet-Pro-DevSecOps/Programmation/GitLab/cardioanalysis/cardiolab/cardiolab/protocols/orthostatic.py:507: PhysiologicalWarning: Recording duration (281 s) is below the recommended minimum of 300 s for resting HRV analysis. Frequency-domain and non-linear metrics may be unreliable.
  features=resting_hrv(standing_rr, method=method),


  ✓  2026-05-17  interp=normal  ΔHR=25.5 bpm  score=0/100

✅  Import orthostatique terminé.


---

## 5 — Supprimer les données d'un utilisateur

Supprime **toutes les lignes** associées à `USER_ID` dans les 8 tables de données.
La table `schema_migrations` et les autres utilisateurs ne sont pas touchés.

> Utile pour nettoyer après un test ou réimporter depuis zéro.
> Passez `CONFIRM_DELETE = True` pour exécuter.

In [ ]:
CONFIRM_DELETE = True  # ← passer à True pour supprimer

_USER_DATA_TABLES = [
    "hrv_features",
    "hrv_orthostatic",
    "hrv_coherence",
    "hrv_hrr",
    "hrv_drift",
    "hrv_vo2max",
    "hrv_raw_sessions",
    "hrv_training_sessions",
]

if not CONFIRM_DELETE:
    print("Suppression désactivée (CONFIRM_DELETE = False).")
    print(f"Données qui seraient supprimées pour '{USER_ID_TEST}' :")
    with HRVRepository.from_env() as repo:
        from psycopg2 import sql as pg_sql
        conn = repo._conn  # noqa: SLF001
        for table in _USER_DATA_TABLES:
            with conn.cursor() as cur:
                cur.execute(
                    pg_sql.SQL("SELECT COUNT(*) FROM {} WHERE user_id = %s;").format(
                        pg_sql.Identifier(table)
                    ),
                    (USER_ID_TEST,),
                )
                n = cur.fetchone()[0]
                print(f"  {table:<28} {n} ligne(s)")
else:
    with HRVRepository.from_env() as repo:
        from psycopg2 import sql as pg_sql
        conn = repo._conn  # noqa: SLF001
        total = 0
        with conn.cursor() as cur:
            for table in _USER_DATA_TABLES:
                cur.execute(
                    pg_sql.SQL("DELETE FROM {} WHERE user_id = %s;").format(
                        pg_sql.Identifier(table)
                    ),
                    (USER_ID_TEST,),
                )
                n = cur.rowcount
                total += n
                print(f"  ✗  {table:<28} {n} ligne(s) supprimée(s)")
        conn.commit()
    print(f"\n✅  {total} ligne(s) supprimée(s) pour '{USER_ID_TEST}'.")

  ✗  hrv_features                 0 ligne(s) supprimée(s)
  ✗  hrv_orthostatic              0 ligne(s) supprimée(s)
  ✗  hrv_coherence                0 ligne(s) supprimée(s)
  ✗  hrv_hrr                      0 ligne(s) supprimée(s)
  ✗  hrv_drift                    0 ligne(s) supprimée(s)
  ✗  hrv_vo2max                   0 ligne(s) supprimée(s)
  ✗  hrv_raw_sessions             0 ligne(s) supprimée(s)
  ✗  hrv_training_sessions        0 ligne(s) supprimée(s)

✅  0 ligne(s) supprimée(s) pour 'demo-user'.
